# Documentação e tags do monitor ao vivo (etapa 5.5)

Aplica **comentário de tabela** e **tags** nos objetos da Fase 5 e confere que **toda coluna tem comentário**. É o que o Catalog Explorer, o Genie e a Fase 6 (governança e LGPD) mostram para quem consulta.

**Objetos**

| Objeto | Tipo | Papel |
|---|---|---|
| `olist.gold.gold_orders_live` | tabela | estado atual de cada pedido, mantido por streaming (visão operacional ao vivo) |
| `olist.gold.vw_pedidos_em_risco` | view | lista de pedidos atrasados e em risco de atraso |
| `olist.ops.streaming_batches` | tabela | um registro por micro-lote do streaming (volume, tardios, latência) |
| `olist.ops.streaming_conferencias` | tabela | resultado das conferências do monitor (replay e idempotência) |
| `olist.quality.validacao_live` | tabela | resultado da validação independente do monitor |

**Tags** (as mesmas da gold, com valores novos)

| Tag | Valores | Significado |
|---|---|---|
| `camada` | `gold`, `ops`, `quality` | camada do objeto |
| `tipo_tabela` | `operacional`, `controle`, `validacao` | `operacional` = visão ao vivo para agir (não é fonte de KPI oficial, ADR 0012); `controle` = registro de execução; `validacao` = resultado de verificação |
| `contem_pii` | `false` | nenhum dos objetos tem dado pessoal: o estado do pedido não guarda cliente, vendedor, cidade nem CEP |

O texto das colunas de `gold_orders_live`, `streaming_batches`, `streaming_conferencias`, `validacao_live` e da view é declarado na criação do objeto; este notebook **não o sobrescreve**, só confere que nenhuma coluna ficou sem comentário. O comentário da tabela e as tags são reaplicados aqui (idempotente).

In [ ]:
# O QUE FAZ: define o catálogo e, para cada objeto da Fase 5, o tipo, o comentário da tabela e as tags, e
#   aplica comentário e tags. Pode rodar várias vezes: sobrescreve o valor anterior.
#
# FUNÇÕES DECLARADAS:
# - texto(valor): troca aspas simples por duas, para o texto caber dentro de um literal SQL.
# - tags_sql(tags): monta o trecho ('chave' = 'valor', ...) de um SET TAGS.
#
# FUNÇÕES NATIVAS E SQL:
# - COMMENT ON TABLE t IS '...': grava o comentário da tabela (não vale para view: o comentário da view
#   já vem do CREATE VIEW).
# - ALTER TABLE t SET TAGS (...) / ALTER VIEW v SET TAGS (...): grava tags no objeto (Unity Catalog).
# - dict.items(): percorre pares (chave, valor). ", ".join(lista): junta os textos com vírgula.
CATALOG = "olist"

OBJETOS = {
    "gold.gold_orders_live": ("TABLE", "gold", "operacional",
        "Estado atual de cada pedido, mantido por streaming (MERGE por event_sequence, deduplicação com watermark pelo instante de chegada). Visão operacional ao vivo: os KPIs oficiais continuam na gold batch (dicionário v1.2, ADR 0012)."),
    "gold.vw_pedidos_em_risco": ("VIEW", "gold", "operacional", None),
    "ops.streaming_batches": ("TABLE", "ops", "controle",
        "Um registro por micro-lote do streaming gold_orders_live: volume, pedidos que avançaram, ignorados por sequência antiga, eventos tardios, latência e duração."),
    "ops.streaming_conferencias": ("TABLE", "ops", "controle",
        "Resultado das conferências do monitor ao vivo (replay com checkpoint novo e idempotência sem arquivo novo), lido pela validação."),
    "quality.validacao_live": ("TABLE", "quality", "validacao",
        "Resultado da validação do monitor ao vivo: estado, micro-lotes, risco e idempotência recalculados direto da silver e da gold batch (etapa 5.4)."),
}


def texto(valor: str) -> str:
    return valor.replace("'", "''")


def tags_sql(tags: dict) -> str:
    return ", ".join(f"'{k}' = '{texto(v)}'" for k, v in tags.items())


for nome, (tipo, camada, tipo_tabela, comentario) in OBJETOS.items():
    t = f"{CATALOG}.{nome}"
    tags = {"camada": camada, "tipo_tabela": tipo_tabela, "contem_pii": "false"}
    if comentario is not None:
        spark.sql(f"COMMENT ON TABLE {t} IS '{texto(comentario)}'")
    spark.sql(f"ALTER {tipo} {t} SET TAGS ({tags_sql(tags)})")
    print(f"{t}: tags {tags}" + (" e comentário aplicados" if comentario else " aplicadas (comentário da view vem do CREATE VIEW)"))

## Conferir

Lê do catálogo o que ficou gravado: as tags de cada objeto, o comentário da tabela e as colunas **sem comentário** (a lista tem que vir vazia para todos os objetos). A célula roda sozinha, numa sessão nova.

In [ ]:
# O QUE FAZ: confere tags, comentário de tabela e comentário de coluna dos objetos da Fase 5. Falha se alguma
#   coluna estiver sem comentário ou se faltar alguma das três tags. Autossuficiente: roda numa sessão nova.
#
# FUNÇÕES NATIVAS:
# - spark.table("olist.information_schema.table_tags"): tags de tabela (schema_name, table_name, tag_name, tag_value).
# - spark.sql("DESCRIBE TABLE t"): col_name, data_type e comment de cada coluna.
# - spark.catalog.getTable(nome).description: comentário da tabela ou da view (None se não houver).
# - set(a) - set(b): itens de a que não estão em b. raise RuntimeError(texto): interrompe com erro.
# - display(df): mostra o DataFrame como tabela interativa.
from pyspark.sql import functions as F

CATALOG = "olist"
OBJETOS = ["gold.gold_orders_live", "gold.vw_pedidos_em_risco", "ops.streaming_batches", "ops.streaming_conferencias", "quality.validacao_live"]
TAGS_ESPERADAS = {"camada", "tipo_tabela", "contem_pii"}

problemas = []
for nome in OBJETOS:
    schema, tabela = nome.split(".")
    completo = f"{CATALOG}.{nome}"

    tags = {r["tag_name"]: r["tag_value"] for r in spark.table(f"{CATALOG}.information_schema.table_tags")
            .where((F.col("schema_name") == schema) & (F.col("table_name") == tabela)).collect()}
    faltando = TAGS_ESPERADAS - set(tags)
    descricao = spark.catalog.getTable(completo).description
    sem_comentario = [r["col_name"] for r in spark.sql(f"DESCRIBE TABLE {completo}").collect()
                      if r["col_name"] and not r["col_name"].startswith("#") and not (r["comment"] or "").strip()]

    print(f"{completo}: tags {tags}; comentário da tabela: {'sim' if descricao else 'NÃO'}; colunas sem comentário: {sem_comentario}")
    if faltando:
        problemas.append(f"{completo}: faltam as tags {sorted(faltando)}")
    if not descricao:
        problemas.append(f"{completo}: sem comentário de tabela")
    if sem_comentario:
        problemas.append(f"{completo}: colunas sem comentário {sem_comentario}")

if problemas:
    raise RuntimeError("documentação incompleta: " + "; ".join(problemas))
print("\ndocumentação conferida: todos os objetos com tags, comentário de tabela e comentário em todas as colunas")